This report predicts the probability that a borrower will experience financial distress within the next two years, using 250,000 historical records pre-split into training (`cs-training.csv`) and test (`cs-test.csv`) sets. The analysis proceeds through a structured diagnostic phase: profiling each feature, visually and mathematically verifying non-normality, scoring skewness to determine pipeline routing, and selecting the correct power transformer per feature.

In [ ]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import normaltest
from great_tables import GT

SEED: int = 42
TARGET: str = "SeriousDlqin2yrs"

random.seed(SEED)
np.random.seed(SEED)

df_train = pd.read_csv("data/cs-training.csv", index_col=0)
df_test  = pd.read_csv("data/cs-test.csv",  index_col=0)

features: list[str] = [c for c in df_train.columns if c != TARGET]

## Missing Value Profiling

Algorithms like SVM and MLP will throw fatal errors on a single `NaN`, so the first step is to quantify any missingness in the training set before any transformation is applied. If missing values are found, understanding the mechanism behind them determines the correct treatment. When data is missing completely at random (MCAR), the gap is unrelated to the underlying values and median imputation is safe. When data is missing not at random (MNAR) — for instance, an unemployed borrower omitting `MonthlyIncome` — the missingness is caused by the value itself, and median imputation introduces bias. In that case, a binary indicator column or model-based imputation is required, and the imputation step must be chained before any downstream mathematical transformation to prevent errors.

In [ ]:
def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Produce a column-level summary: dtype, null count, null %, and basic stats.

    Args:
        df: The DataFrame to profile.

    Returns:
        Summary DataFrame with one row per column.
    """
    null_count = df.isnull().sum()
    null_pct   = (null_count / len(df) * 100).round(2)
    return pd.DataFrame({
        "dtype"     : df.dtypes,
        "null_count": null_count,
        "null_%"    : null_pct,
        "mean"      : df.mean(numeric_only=True).round(4),
        "std"       : df.std(numeric_only=True).round(4),
        "min"       : df.min(numeric_only=True),
        "max"       : df.max(numeric_only=True),
    })


GT(profile_dataframe(df_train).reset_index().rename(columns={"index": "feature"}))

In [ ]:
def test_mcar(df: pd.DataFrame, cols: list[str], target: str) -> pd.DataFrame:
    """Test whether missingness in each column is associated with the target variable.

    A statistically significant point-biserial correlation between a missing
    indicator and the target is evidence against MCAR, suggesting MAR or MNAR.

    Args:
        df: Training DataFrame.
        cols: Columns to test for informative missingness.
        target: Binary target column name.

    Returns:
        DataFrame with default rates, correlation, and p-value per column.
    """
    from scipy.stats import pointbiserialr

    rows = []
    for col in cols:
        indicator = df[col].isna().astype(int)
        r, p = pointbiserialr(indicator, df[target])
        rows.append({
            "column": col,
            "missing_pct": round(indicator.mean() * 100, 2),
            "default_rate_observed": round(df.loc[indicator == 0, target].mean(), 4),
            "default_rate_missing": round(df.loc[indicator == 1, target].mean(), 4),
            "r": round(r, 4),
            "p_value": round(p, 10),
        })
    return pd.DataFrame(rows)

missing_cols = [c for c in features if df_train[c].isna().any()]
mcar_df = test_mcar(df_train, missing_cols, TARGET)
(
    GT(mcar_df)
    .tab_header(
        title="MCAR Test — Point-Biserial Correlation",
        subtitle="Is missingness associated with the target variable?"
    )
    .fmt_number(columns=["missing_pct", "default_rate_observed", "default_rate_missing", "r"], decimals=4)
    .fmt_scientific(columns="p_value")
)

Both columns return p-values far below 0.05, so we reject MCAR for both. Missingness is statistically associated with the target variable, confirming MNAR.

What makes this a genuine surprise is the direction: rows with missing values show a lower default rate, not higher. The intuitive assumption would be the opposite — that missing income signals financial instability and therefore higher risk. Instead, the data suggests the missing group may skew toward retired borrowers: no employment income to report, but also lower credit risk. The absence of a value is itself a meaningful signal, just not the one we expected.

Median imputation alone would silently discard this signal. The correct treatment is to first create a binary `_is_missing` flag for each affected column — preserving the information that absence of a value is predictive — and then impute the median to satisfy algorithm requirements.

We now apply the two-step fix derived above, creating the binary flags before imputation so no signal is overwritten by the fill.

In [ ]:
MONTHLY_INCOME_MEDIAN: float = df_train["MonthlyIncome"].median()
DEPENDENTS_MEDIAN: float = df_train["NumberOfDependents"].median()


def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Add boolean missing-value indicator columns before imputation.

    Flags are derived before filling so the signal is not overwritten.
    Uses integer encoding (0/1) for direct compatibility with sklearn estimators.

    Args:
        df: DataFrame to annotate.

    Returns:
        DataFrame with IsMonthlyIncomeMissing and IsNumberOfDependentsMissing appended.
    """
    return df.assign(
        IsMonthlyIncomeMissing=df["MonthlyIncome"].isna().astype(int),
        IsNumberOfDependentsMissing=df["NumberOfDependents"].isna().astype(int),
    )


def impute_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values using training-set medians.

    MonthlyIncome is filled with the training median.
    NumberOfDependents is filled with 0, which equals the training median
    (59% of observed values are 0) and aligns with the retiree hypothesis.

    Args:
        df: DataFrame with missing-value flag columns already added.

    Returns:
        DataFrame with NaNs in MonthlyIncome and NumberOfDependents resolved.
    """
    return df.fillna({
        "MonthlyIncome": MONTHLY_INCOME_MEDIAN,
        "NumberOfDependents": DEPENDENTS_MEDIAN,
    })


df_train = impute_missing(add_missing_flags(df_train))
features = [c for c in df_train.columns if c != TARGET]
df_train[["MonthlyIncome", "IsMonthlyIncomeMissing",
          "NumberOfDependents", "IsNumberOfDependentsMissing"]].describe().T

## Class Imbalance — Target Variable

We examine the distribution of `SeriousDlqin2yrs` to understand how the two outcomes are represented in the training set. A large disparity between classes means a naive model could achieve high accuracy by always predicting the majority class, while having zero predictive power for actual defaults. If severe imbalance is detected, two remedies are available. SMOTE synthesises minority-class examples in the training fold to balance the class ratio, while passing `class_weight='balanced'` to sklearn estimators weights the minority class inversely proportional to its frequency — a lower-cost option that requires no synthetic data. Either strategy must be applied only on the training set to prevent leakage into the test evaluation.

In [ ]:
def check_class_balance(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """Compute class distribution for the target variable.

    Args:
        df: Training DataFrame.
        target: Name of the target column.

    Returns:
        DataFrame with class label, count, and percentage.
    """
    counts = df[target].value_counts().rename_axis("class").reset_index(name="count")
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(2)
    return counts

class_balance_df = check_class_balance(df_train, TARGET)
(
    GT(class_balance_df)
    .tab_header(title="Class Distribution", subtitle=TARGET)
    .fmt_integer(columns="count")
    .fmt_number(columns="pct", decimals=2)
)

## Distribution Analysis

### Visual Check

Histograms for every continuous feature give us a first look at each feature's shape before any formal testing. We are looking for heavy right tails, narrow spikes at zero, or multi-modal shapes — any of which indicate deviation from normality that could destabilise distance-based and gradient-descent models.

In [ ]:
df_train[features].hist(bins=60, figsize=(15, 10), layout=(4, 3))
plt.suptitle("Feature Distributions — Training Set", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### Mathematical Proof (D'Agostino K²)

To formally test whether each feature follows a normal distribution, we apply D'Agostino's $K^2$ test, which combines skewness and kurtosis into a single statistic. The null hypothesis $H_0$ is that the feature originates from a normal distribution. A p-value below 0.05 gives grounds to reject $H_0$, confirming the feature is statistically non-normal. Features where we fail to reject may not require power transformation. If non-normality is widespread, standard scaling alone will be insufficient and the pipeline must rely on power transformations to address the underlying distribution shape.

In [ ]:
def run_normality_tests(df: pd.DataFrame, features: list[str], alpha: float = 0.05) -> pd.DataFrame:
    """Run D'Agostino's K² normality test on each feature and report results.

    Args:
        df:       DataFrame containing the features.
        features: Column names to test.
        alpha:    Significance level for rejecting H0 (default 0.05).

    Returns:
        DataFrame with columns: feature, statistic, p_value, reject_H0, verdict.
    """
    records = [
        {
            "feature"  : col,
            "statistic": round(stat, 4),
            "p_value"  : round(p, 6),
            "reject_H0": p < alpha,
            "verdict"  : "Non-normal" if p < alpha else "Normal",
        }
        for col in features
        for stat, p in [normaltest(df[col].dropna())]
    ]
    return pd.DataFrame(records)


GT(run_normality_tests(df_train, features))

### Data Quality — DebtRatio Encoding Inconsistency

`DebtRatio` is defined as monthly debt payments divided by monthly income. If `MonthlyIncome` is missing, a valid ratio should be uncomputable — yet every row with missing income still carries a non-null `DebtRatio`. We examine whether the column is encoding something different for those rows by comparing its distribution between the two groups. A large divergence would suggest the column has mixed semantics.

In [ ]:
def compare_groups_by_missingness(
    df: pd.DataFrame, flag_col: str, compare_features: list[str]
) -> pd.DataFrame:
    """Compare feature medians between rows where flag_col is 1 vs 0.

    Args:
        df: Training DataFrame.
        flag_col: Binary indicator column (1 = was missing, 0 = was present).
        compare_features: Features to summarise across the two groups.

    Returns:
        DataFrame with per-feature medians for each group and their ratio.
    """
    is_missing = df[flag_col].astype(bool)
    rows = []
    for col in compare_features:
        med_present = df.loc[~is_missing, col].median()
        med_missing = df.loc[is_missing, col].median()
        ratio = (
            round(med_missing / med_present, 2)
            if med_present not in (0, None) and pd.notna(med_present) and pd.notna(med_missing)
            else None
        )
        rows.append({
            "feature": col,
            "median_income_present": med_present,
            "median_income_missing": med_missing,
            "ratio": ratio,
        })
    return (
        pd.DataFrame(rows)
        .sort_values("ratio", ascending=False, na_position="last")
        .reset_index(drop=True)
    )

non_target_features = [c for c in df_train.columns if c != TARGET]
group_df = compare_groups_by_missingness(df_train, "IsMonthlyIncomeMissing", non_target_features)
(
    GT(group_df)
    .tab_header(
        title="Feature Medians — MonthlyIncome Missing vs. Present",
        subtitle="Sorted by ratio (missing / present) descending — ratio column omitted where present median is 0"
    )
    .fmt_number(columns=["median_income_present", "median_income_missing", "ratio"], decimals=4)
)

In [ ]:
def plot_debtratio_before_after(df: pd.DataFrame, flag_col: str, col: str = "DebtRatio") -> None:
    """Plot DebtRatio distribution before and after isolating the corrupted rows.

    Args:
        df: Training DataFrame with binary missing-income flag.
        flag_col: Binary indicator for missing income (1 = missing).
        col: Feature column to plot.
    """
    is_corrupted = df[flag_col].astype(bool)
    before = df[col]
    after  = df.loc[~is_corrupted, col]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(
        "DebtRatio — Before and After Removing Corrupted Rows",
        fontsize=12, fontweight="bold"
    )

    axes[0].hist(before.clip(upper=before.quantile(0.995)), bins=80, color="steelblue", alpha=0.8)
    axes[0].set_title("All rows (clipped at 99.5th percentile for visibility)")
    axes[0].set_xlabel("DebtRatio")
    axes[0].set_ylabel("count")

    axes[1].hist(after.clip(upper=5), bins=80, color="darkorange", alpha=0.8)
    axes[1].set_title("Income-present rows only (clipped at 5 for visibility)")
    axes[1].set_xlabel("DebtRatio")

    plt.tight_layout()
    plt.show()


plot_debtratio_before_after(df_train, "IsMonthlyIncomeMissing")

`DebtRatio` is the only feature with a pathological divergence — its median is roughly 3,900 times higher in the missing-income group (1,159 vs 0.296). While a debt-to-income ratio above 1.0 is technically possible, values in the hundreds or thousands are not interpretable as any ratio. The 75th percentile sits at 0.87, then the 90th jumps to 1,267 — that is not a heavy tail, it is two irreconcilable encodings in the same column. For the missing-income group, the column almost certainly stores raw monthly debt obligations in dollars rather than a computed ratio.

The broader pattern reinforces the retired-borrower hypothesis: the missing-income group is older (median age 57 vs 51) and carries lower revolving utilisation (0.08 vs 0.18). Since `IsMonthlyIncomeMissing` already captures the structural difference of these rows, the cleanest action is to drop `DebtRatio` entirely — removing a corrupted feature without discarding any information not already represented more reliably elsewhere.

### Data Quality — RevolvingUtilization Extreme Values

`RevolvingUtilizationOfUnsecuredLines` should be a ratio between 0 and 1, representing credit used divided by credit limit. Values marginally above 1.0 are plausible, since fees and interest can push balances past the stated limit, but we inspect the upper tail for the same dollar-encoding issue found in `DebtRatio`. A cliff in the percentile distribution — where the column transitions from ratio-scale values to implausibly large numbers — would confirm the problem.

In [ ]:
def profile_upper_tail(
    df: pd.DataFrame, col: str, percentiles: list[float] | None = None
) -> pd.DataFrame:
    """Summarise the upper tail of a column to surface encoding discontinuities.

    Args:
        df: Training DataFrame.
        col: Column to profile.
        percentiles: Percentile breakpoints to include. Defaults to standard set.

    Returns:
        DataFrame of percentile labels and their corresponding values.
    """
    pcts = percentiles or [0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1.0]
    labels = [f"p{int(p * 100)}" if p < 1 else "max" for p in pcts]
    values = df[col].quantile(pcts).values
    return pd.DataFrame({"percentile": labels, "value": values})

tail_df = profile_upper_tail(df_train, "RevolvingUtilizationOfUnsecuredLines")
(
    GT(tail_df)
    .tab_header(
        title="RevolvingUtilizationOfUnsecuredLines — Upper Tail",
        subtitle="Identifying the encoding cliff"
    )
    .fmt_number(columns="value", decimals=4)
)

The cliff is unmistakable. The 90th percentile is 0.98, the 95th is 1.0, and the 99.9th jumps to 1,571 with a maximum of 50,708. Values marginally above 1.0 are defensible, but values in the thousands are dollar balances, not ratios — the same encoding inconsistency found in `DebtRatio`. Unlike that column, however, the feature is well-behaved for the vast majority of rows, so dropping it would discard real signal. Capping at the 99th percentile (~1.09) preserves legitimate near-limit values while suppressing the dollar-encoded outliers.

### Data Quality — Delinquency Column Error Codes

The three delinquency count columns showed a suspicious gap in their value distributions — counts jump from plausible values (0–17) directly to 96 and 98, with nothing in between. In financial survey datasets, values such as 96, 98, and 99 are commonly used as error codes to flag records as not applicable or erroneous rather than genuine counts. We investigate whether the same rows carry these anomalous values across all three columns simultaneously, which would confirm they are error codes rather than real delinquency counts.

In [ ]:
def find_error_code_rows(
    df: pd.DataFrame, cols: list[str], error_codes: list[int]
) -> pd.DataFrame:
    """Identify rows carrying error codes and verify cross-column consistency.

    Args:
        df: Training DataFrame.
        cols: Columns expected to share the error code encoding.
        error_codes: Candidate error codes to check.

    Returns:
        DataFrame summarising error code counts and cross-column consistency.
    """
    rows = []
    for val in error_codes:
        masks = [df[c] == val for c in cols]
        count = masks[0].sum()
        all_identical = all((masks[0] == m).all() for m in masks[1:])
        rows.append({
            "error_code": val,
            "row_count": int(count),
            "identical_across_all_columns": all_identical,
        })
    return pd.DataFrame(rows)

delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
error_code_df = find_error_code_rows(df_train, delinquency_cols, error_codes=[96, 98])
(
    GT(error_code_df)
    .tab_header(
        title="Delinquency Column Error Codes",
        subtitle="Do the same rows carry the same anomalous value across all three columns?"
    )
    .fmt_integer(columns=["error_code", "row_count"])
)

Both error codes (96 and 98) are carried by exactly the same rows across all three delinquency columns. A real borrower cannot have identical counts of 30–59, 60–89, and 90+ day delinquencies, since these are sequential severity buckets. This conclusively identifies them as error codes. With only 269 affected rows representing 0.18% of the training set, the cleanest treatment is to replace these values with `NaN` and drop the affected rows outright — imputing delinquency counts for corrupted records would risk introducing more noise than signal.

### Data Quality — MonthlyIncome Extreme Values

`MonthlyIncome` has a maximum of \$3,008,750 per month, roughly 38 times the 99.9th percentile. Before accepting these as legitimate, we check whether extreme income values are associated with lower default rates and whether they are concentrated enough to distort downstream transformations. We are looking both for a clear relationship between income and default risk and for a distributional cliff similar to those found in `DebtRatio` and `RevolvingUtilization`.

In [ ]:
def default_rate_by_income_bucket(
    df: pd.DataFrame, income_col: str, target: str
) -> pd.DataFrame:
    """Compute default rate, row count, and median income across income brackets.

    Args:
        df: Training DataFrame.
        income_col: Name of the income column.
        target: Binary target column name.

    Returns:
        DataFrame with one row per income bracket.
    """
    df_valid = df.dropna(subset=[income_col]).copy()
    df_valid["income_bucket"] = pd.cut(
        df_valid[income_col],
        bins=[0, 2_000, 5_000, 10_000, 20_000, 50_000, float("inf")],
        labels=["<$2k", "$2k–$5k", "$5k–$10k", "$10k–$20k", "$20k–$50k", ">$50k"],
    )
    return (
        df_valid.groupby("income_bucket", observed=True)
        .agg(
            row_count=(income_col, "count"),
            median_income=(income_col, "median"),
            default_rate=(target, "mean"),
        )
        .round(4)
        .reset_index()
    )

income_bucket_df = default_rate_by_income_bucket(df_train, "MonthlyIncome", TARGET)
(
    GT(income_bucket_df)
    .tab_header(
        title="Default Rate by Monthly Income Bracket",
        subtitle="Rows with missing MonthlyIncome excluded"
    )
    .fmt_integer(columns="row_count")
    .fmt_currency(columns="median_income", currency="USD")
    .fmt_percent(columns="default_rate", decimals=2)
)

Higher income does reduce default risk, but not monotonically. The default rate falls sharply from 9.1% at under \$2k per month to 4.2% at \$10k–\$20k, then slightly increases for the highest earners (5.3% at \$20k–\$50k, 5.7% above \$50k). Very high earners may carry more leverage or take on riskier debt structures, so the assumption that wealth implies safety does not hold universally at the extreme end.

The extreme values are almost certainly real rather than encoding errors — they are demographically coherent and all non-defaulting. However, a handful of values 38 times the 99.9th percentile will dominate the Yeo-Johnson lambda fit for Branch B models. For tree models in Branch A, no cap is needed since trees split on thresholds and handle extreme values natively. For Branch B, `MonthlyIncome` is capped at the 99.9th percentile (~\$78k/month) before transformation, applied only after the `IsMonthlyIncomeMissing` flag and imputation steps are complete.

## Multicollinearity — Correlation Check

We compute the Pearson correlation matrix across all features and inspect the top 10 most correlated pairs. Highly collinear pairs are measuring the same underlying signal, so retaining both adds noise without adding information. Pairs with $|r| > 0.8$ are candidates for removal. For SVM and MLP, redundant features cause computational bloat and increase overfitting risk. For tree models, feature importance scores become artificially diluted across collinear features, rendering them uninterpretable. The appropriate response is to drop the redundant feature from each flagged pair, retaining the more informative signal.

In [ ]:
def top_correlations(
    df: pd.DataFrame, features: list[str], n: int = 10
) -> pd.DataFrame:
    """Return the top N most correlated feature pairs by absolute Pearson r.

    Args:
        df: Training DataFrame.
        features: Feature columns to evaluate.
        n: Number of top pairs to return.

    Returns:
        DataFrame of feature pairs sorted by absolute correlation, descending.
    """
    corr = df[features].corr(method="pearson")
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ["feature_a", "feature_b", "correlation"]
    return (
        pairs.reindex(pairs["correlation"].abs().sort_values(ascending=False).index)
        .head(n)
        .reset_index(drop=True)
    )

top_corr_df = top_correlations(df_train, features)
(
    GT(top_corr_df)
    .tab_header(title="Top 10 Feature Correlations", subtitle="Pearson r — sorted by absolute value")
    .fmt_number(columns="correlation", decimals=4)
)

## Skewness Heuristic — Pipeline Branching

To programmatically route features into the correct pipeline branch, we compute the absolute skewness score for each feature and apply a threshold rule: features with $|\text{skew}| > 1$ are considered highly skewed and routed to Branch B for transformation and scaling, while those at or below the threshold are considered approximately symmetric and routed to Branch A to be passed raw and unscaled. Features with low skew scores can safely bypass power transformation; those above the threshold require treatment before distance-based or gradient-descent models can use them effectively.

In [ ]:
def compute_skewness_table(df: pd.DataFrame, features: list[str], threshold: float = 1.0) -> pd.DataFrame:
    """Compute skewness for each feature and assign a pipeline branch.

    Args:
        df:        DataFrame containing the features.
        features:  Column names to evaluate.
        threshold: Absolute skew above which a feature is flagged (default 1.0).

    Returns:
        DataFrame sorted by abs_skew descending.
    """
    records = [
        {
            "feature"      : col,
            "skewness"     : round(df[col].skew(), 4),
            "abs_skew"     : round(abs(df[col].skew()), 4),
            "highly_skewed": abs(df[col].skew()) > threshold,
            "branch"       : "B — Transform + Scale" if abs(df[col].skew()) > threshold else "A — Raw (tree-safe)",
        }
        for col in features
    ]
    return pd.DataFrame(records).sort_values("abs_skew", ascending=False)


skewness_df = compute_skewness_table(df_train, features)
GT(skewness_df)

## Transformer Selection — Box-Cox vs. Yeo-Johnson

Power transformers have strict domain requirements, so before applying any transformation we inspect the minimum value of each Branch B feature to select the correct algorithm. The decision rule is: if $\min(\text{feature}) > 0$, use Box-Cox; if $\min(\text{feature}) \le 0$, use Yeo-Johnson.

While Yeo-Johnson safely handles negatives and zeros, defaulting to it unconditionally carries real trade-offs. It optimises a complex piecewise function that is slower on large datasets. Box-Cox lambdas map to interpretable classic transforms — $\lambda = 0$ is exactly a log transform — whereas Yeo-Johnson's $+1$ shift obscures the underlying adjustment. For strictly positive data, Box-Cox also tends to find a mathematically tighter normal fit. The domain check here is not a formality: it determines the strictly correct algorithm.

In [ ]:
def select_transformer(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Assign a power transformer to each feature based on its minimum value.

    Decision rule:
      min > 0  + Box-Cox (strictly positive data)
      min <= 0 + Yeo-Johnson (handles zeros and negatives)

    Args:
        df:       DataFrame containing the features.
        features: Column names to evaluate (typically the highly-skewed subset).

    Returns:
        DataFrame with columns: feature, min_value, transformer, reason.
    """
    records = [
        {
            "feature"    : col,
            "min_value"  : df[col].min(),
            "transformer": "Box-Cox" if df[col].min() > 0 else "Yeo-Johnson",
            "reason"     : "min > 0 — strictly positive" if df[col].min() > 0
                           else "min ≤ 0 — contains zeros or negatives",
        }
        for col in features
    ]
    return pd.DataFrame(records)


highly_skewed_features = skewness_df.loc[skewness_df["highly_skewed"], "feature"].tolist()
transformer_plan = select_transformer(df_train, highly_skewed_features)
GT(transformer_plan)

## Pipeline

The diagnosis established four data quality treatments and a clear branching strategy, all of which are encoded here into a reproducible sklearn pipeline fitted strictly on the training set. For both branches, error codes (96 and 98) are replaced with `NaN` in the delinquency columns and the affected rows dropped, `DebtRatio` is dropped due to its corrupted dollar-amount encoding, the two delinquency columns with near-perfect collinearity with `NumberOfTimes90DaysLate` are dropped, and `RevolvingUtilizationOfUnsecuredLines` is capped at the 99.9th percentile. Branch A passes the cleaned features through raw and unscaled for the tree models. Branch B additionally caps `MonthlyIncome` at the 99.9th percentile before applying Yeo-Johnson followed by StandardScaler to all skewed features.

In [ ]:
#| echo: true
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer, StandardScaler
from typing import Protocol
from sklearn.base import BaseEstimator
from enum import Enum, auto


class FeatureTransformer(Protocol):
    """Interface for per-feature Branch B transformations."""

    def __str__(self) -> str: ...
    def pipeline(self) -> BaseEstimator: ...


class BinaryPassthrough(FeatureTransformer):
    """Identity transform for binary indicator features — preserves 0/1 encoding unchanged."""

    def __str__(self) -> str:
        return "passthrough"

    def pipeline(self) -> FunctionTransformer:
        return FunctionTransformer()


class SymmetricTransformer(FeatureTransformer):
    """Standard scaling only, for low-skew continuous features that need centering but not reshaping."""

    def __str__(self) -> str:
        return "StandardScaler"

    def pipeline(self) -> StandardScaler:
        return StandardScaler()


class SkewedTransformer(FeatureTransformer):
    """Yeo-Johnson followed by standard scaling, for highly skewed features without extreme outliers."""

    def __str__(self) -> str:
        return "Yeo-Johnson + StandardScaler"

    def pipeline(self) -> Pipeline:
        return Pipeline([
            ("power", PowerTransformer(method="yeo-johnson")),
            ("scaler", StandardScaler()),
        ])


class CappedSkewedTransformer(SkewedTransformer):
    """Extends SkewedTransformer by prepending a percentile cap for features with extreme outliers.

    Args:
        cap: Upper bound to clip values to before the Yeo-Johnson transform.
    """

    def __init__(self, *, cap: float) -> None:
        self.cap = cap

    def __str__(self) -> str:
        return "cap \u2192 Yeo-Johnson + StandardScaler"

    def pipeline(self) -> Pipeline:
        return Pipeline([
            ("cap", FunctionTransformer(lambda X: np.clip(X, None, self.cap))),
            *super().pipeline().steps,
        ])




REVOLVING_CAP: float = df_train["RevolvingUtilizationOfUnsecuredLines"].quantile(0.999)
INCOME_CAP: float = df_train["MonthlyIncome"].quantile(0.999)
FEATURE_PIPELINE_MAP: dict[str, FeatureTransformer] = {
    "RevolvingUtilizationOfUnsecuredLines": CappedSkewedTransformer(cap=REVOLVING_CAP),
    "age":                                  SymmetricTransformer(),
    "MonthlyIncome":                        CappedSkewedTransformer(cap=INCOME_CAP),
    "NumberOfOpenCreditLinesAndLoans":      SkewedTransformer(),
    "NumberOfTimes90DaysLate":              SkewedTransformer(),
    "NumberRealEstateLoansOrLines":         SkewedTransformer(),
    "NumberOfDependents":                   SkewedTransformer(),
    "IsMonthlyIncomeMissing":               BinaryPassthrough(),
    "IsNumberOfDependentsMissing":          BinaryPassthrough(),
}

df_clean = (
    df_train
    .assign(**{"NumberOfTime30-59DaysPastDueNotWorse": lambda x: x["NumberOfTime30-59DaysPastDueNotWorse"].replace(96, np.nan)})
    .assign(**{"NumberOfTime30-59DaysPastDueNotWorse": lambda x: x["NumberOfTime30-59DaysPastDueNotWorse"].replace(98, np.nan)})
    .assign(**{"NumberOfTime60-89DaysPastDueNotWorse": lambda x: x["NumberOfTime60-89DaysPastDueNotWorse"].replace(96, np.nan)})
    .assign(**{"NumberOfTime60-89DaysPastDueNotWorse": lambda x: x["NumberOfTime60-89DaysPastDueNotWorse"].replace(98, np.nan)})
    .assign(NumberOfTimes90DaysLate=lambda x: x["NumberOfTimes90DaysLate"].replace(96, np.nan))
    .assign(NumberOfTimes90DaysLate=lambda x: x["NumberOfTimes90DaysLate"].replace(98, np.nan))
    .dropna(subset=["NumberOfTimes90DaysLate"])
    .reset_index(drop=True)
)

In [ ]:
(
    GT(pd.DataFrame([
        {"feature": col, "branch_a": "raw", "branch_b": str(tx)}
        for col, tx in FEATURE_PIPELINE_MAP.items()
    ]))
    .tab_header(title="Feature Routing", subtitle="Treatment applied per branch")
)

In [ ]:
X_train = df_clean[list(FEATURE_PIPELINE_MAP.keys())]

branch_b_preprocessor = ColumnTransformer(
    transformers=[(col, tx.pipeline(), [col]) for col, tx in FEATURE_PIPELINE_MAP.items()],
    remainder="drop",
)

_X_scaled = pd.DataFrame(
    branch_b_preprocessor.fit_transform(X_train),
    columns=list(FEATURE_PIPELINE_MAP.keys()),
)
_y = df_clean[TARGET]


class Data(Enum):
    """Training dataset variants keyed by preprocessing branch.

    Each member exposes .X (feature DataFrame) and .y (target Series),
    assigned after construction to avoid DataFrame hashability constraints.
    """
    CLEAN  = auto()
    SCALED = auto()


Data.CLEAN.X,  Data.CLEAN.y  = X_train,   _y
Data.SCALED.X, Data.SCALED.y = _X_scaled, _y

In [ ]:
(
    GT(pd.DataFrame([
        {"branch": "A — Tree models", "rows": Data.CLEAN.X.shape[0],  "features": Data.CLEAN.X.shape[1],  "transformations": "none (raw)"},
        {"branch": "B — SVM / MLP",   "rows": Data.SCALED.X.shape[0], "features": Data.SCALED.X.shape[1], "transformations": "per feature routing"},
    ]))
    .tab_header(title="Pipeline Output", subtitle="Training set dimensions per branch")
    .fmt_integer(columns=["rows", "features"])
)

In [ ]:
def plot_branch_b_distributions(
    df_before: pd.DataFrame,
    df_after: pd.DataFrame,
    features: list[str],
) -> None:
    """Plot Branch B feature distributions before and after pipeline transformation.

    Args:
        df_before: Cleaned but untransformed feature DataFrame.
        df_after: Transformed Branch B DataFrame.
        features: Features to compare — should be the skewed Branch B columns.
    """
    n = len(features)
    fig, axes = plt.subplots(n, 2, figsize=(14, n * 2.5))
    fig.suptitle(
        "Branch B Feature Distributions — Before vs. After Pipeline",
        fontsize=13, fontweight="bold", y=1.01,
    )
    for i, feat in enumerate(features):
        axes[i, 0].hist(df_before[feat].dropna(), bins=60, color="steelblue", alpha=0.8)
        axes[i, 0].set_title(f"{feat}  |  before", fontsize=9)
        axes[i, 0].set_ylabel("count")

        axes[i, 1].hist(df_after[feat], bins=60, color="darkorange", alpha=0.8)
        axes[i, 1].set_title(f"{feat}  |  after Yeo-Johnson + StandardScaler", fontsize=9)

    plt.tight_layout()
    plt.show()


transformed_features = [col for col, tx in FEATURE_PIPELINE_MAP.items() if isinstance(tx, SkewedTransformer)]
plot_branch_b_distributions(Data.CLEAN.X, Data.SCALED.X, transformed_features)

## Summary

The sections above establish a complete diagnostic picture of the training data. Features with $|\text{skew}| \le 1$ are passed raw and unscaled to the tree models in Branch A. Highly skewed features are imputed, power-transformed, and scaled for the distance and gradient-based models in Branch B. All imputers, transformers, and scalers are fitted only on the training set and applied via `.transform()` to the test set — `.fit()` and `.fit_transform()` are never called on test data.